# Notebook 3 — Benchmark ML — Paradigme 2 : CVSS + EPSS (Threat Intelligence)

## Objectif

Ce notebook injecte les scores **EPSS** (*Exploit Prediction Scoring System*,  
Jacobs et al. 2021) comme features d'entrée du modèle, puis compare les résultats  
avec EPSS utilisé seul comme référence état de l'art.

**Il quantifie la valeur informationnelle de la Threat Intelligence exogène**  
(question de recherche 2 du mémoire).

**Dépendance :** `data/api_vulnerabilities_processed.csv` généré par le Notebook 1.  
**Connexion Internet requise** : téléchargement du snapshot EPSS journalier.

**Architecture des features (Paradigme 2) :**  
11 features = 8 OHE CVSS + 1 temporelle (`cve_age_days`) + 2 EPSS (`epss_score`, `percentile`)

**Résultats attendus :**
- AUC ≈ 0.99 (CVSS + EPSS)
- EPSS standalone ≈ 0.99 (référence Jacobs et al.)
- Δ gain vs Paradigme 1 : +0.29 AUC

---
> **Note méthodologique :** la Phase 7 n'est pas une comparaison indépendante  
> contre EPSS — notre modèle *utilise* epss_score comme feature.  
> Elle illustre la valeur ajoutée de combiner CVSS et EPSS par rapport à EPSS seul.

In [ ]:
# ==============================================================================
# CELLULE 1 — Imports, chemins et chargement du dataset
# ==============================================================================

import os
import gzip
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

warnings.filterwarnings('ignore')

# ── Résolution du répertoire de données ──────────────────────────────────────
_cwd = Path(os.getcwd())
DATA_DIR   = (_cwd.parent / 'data') if _cwd.name == 'src' else (_cwd / 'data')
INPUT_PATH = DATA_DIR / 'api_vulnerabilities_processed.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'Fichier introuvable : {INPUT_PATH}\n'
        'Exécutez d abord le Notebook 1 (01_collecte_donnees.ipynb).'
    )

df = pd.read_csv(INPUT_PATH)
assert df.shape[1] >= 13, f'CSV dégradé : {df.shape[1]} colonnes'

print(f'[✓] Dataset chargé : {df.shape[0]:,} CVE × {df.shape[1]} colonnes')
print(f'[+] Y=1 (exploitées) : {df["is_exploited"].sum():,} '
      f'({df["is_exploited"].mean()*100:.2f} %)')

# ── Paramètres communs ────────────────────────────────────────────────────────
EPSS_URL         = 'https://epss.cyentia.com/epss_scores-current.csv.gz'
FEATURES_TO_ENCODE = ['attack_vector', 'attack_complexity',
                       'privileges_required', 'user_interaction', 'scope']
RANDOM_STATE     = 42
TEST_SIZE        = 0.25

## Section 1 — Injection des Scores EPSS (Threat Intelligence exogène)

Les scores EPSS encodent la probabilité journalière qu'une CVE soit exploitée  
dans les 30 jours, calculée à partir de signaux non accessibles au NVD :  
scans actifs (Shadowserver), PoC publics (GitHub/Exploit-DB), présence dans  
Metasploit, activité médiatique et threat intelligence mondiale.

**Source :** Cyentia Institute / FIRST.org — snapshot journalier public  
**Référence :** Jacobs J. et al. (2021). *Exploit Prediction Scoring System (EPSS)*

In [ ]:
# ==============================================================================
# CELLULE 2 — Téléchargement et injection du snapshot EPSS
# ==============================================================================

epss_ok = False

print('[+] Téléchargement du snapshot EPSS journalier...')
try:
    resp = requests.get(EPSS_URL, timeout=60)
    resp.raise_for_status()

    with gzip.open(io.BytesIO(resp.content)) as gz:
        epss_df = pd.read_csv(gz, comment='#', header=0,
                              names=['cve_id', 'epss_score', 'percentile'])

    epss_df['epss_score'] = pd.to_numeric(epss_df['epss_score'], errors='coerce')
    epss_df['percentile'] = pd.to_numeric(epss_df['percentile'],  errors='coerce')

    print(f'[✓] Snapshot EPSS chargé : {len(epss_df):,} CVE scorées')

    # ── Fusion avec le dataset NVD (left join pour conserver toutes les CVE) ─
    n_before = len(df)
    df = df.merge(epss_df[['cve_id', 'epss_score', 'percentile']],
                  on='cve_id', how='left')
    assert len(df) == n_before, 'La jointure a modifié le nombre de lignes'

    n_miss = df['epss_score'].isna().sum()
    df['epss_score'] = df['epss_score'].fillna(0.0)
    df['percentile'] = df['percentile'].fillna(0.0)

    print(f'[+] Couverture EPSS : {n_before - n_miss:,}/{n_before:,} CVE scorées '
          f'({n_miss} imputées à 0.0)')

    # ── Analyse du signal discriminant EPSS ───────────────────────────────────
    mu_exp = df.loc[df['is_exploited'] == 1, 'epss_score'].mean()
    mu_all = df['epss_score'].mean()
    ratio  = mu_exp / mu_all if mu_all > 0 else float('inf')

    print()
    print('─' * 50)
    print('SIGNAL DISCRIMINANT EPSS (avant modélisation)')
    print('─' * 50)
    print(f'  EPSS moyen — exploitées (Y=1) : {mu_exp:.4f}')
    print(f'  EPSS moyen — corpus global    : {mu_all:.4f}')
    print(f'  Ratio                         : {ratio:.0f}×  ← signal dominant')
    print()
    epss_ok = True

except Exception as e:
    print(f'[!] EPSS indisponible ({e})')
    print('    Le notebook va s exécuter en mode Paradigme 1 (CVSS seul).')
    print('    Reconnectez-vous à Internet et ré-exécutez cette cellule.')

paradigme = 'PARADIGME 2 — CVSS + EPSS' if epss_ok else 'PARADIGME 1 — CVSS seul (EPSS absent)'
print(f'[→] {paradigme}')

## Section 2 — Feature Engineering et Partitionnement

In [ ]:
# ==============================================================================
# CELLULE 3 — Construction de la matrice X (OHE + temporal + EPSS)
# ==============================================================================

df_model = df.dropna(subset=FEATURES_TO_ENCODE).copy()
Y = df_model['is_exploited'].values

# ── One-Hot Encoding des features CVSS catégorielles ─────────────────────────
X_encoded = pd.get_dummies(
    df_model[FEATURES_TO_ENCODE],
    columns=FEATURES_TO_ENCODE,
    drop_first=True,
    dtype=float
)

# ── Feature temporelle ────────────────────────────────────────────────────────
if 'cve_age_days' in df_model.columns:
    X_encoded['cve_age_days'] = df_model['cve_age_days'].values

# ── Features EPSS (activées uniquement si le téléchargement a réussi) ─────────
if epss_ok:
    X_encoded['epss_score'] = df_model['epss_score'].values
    X_encoded['percentile'] = df_model['percentile'].values

n_epss = 2 if epss_ok else 0
print(f'[+] Matrice X : {X_encoded.shape[1]} features')
print(f'    8 OHE CVSS + 1 temporal + {n_epss} EPSS')
print(f'    {list(X_encoded.columns)}')

# ── Partitionnement stratifié ─────────────────────────────────────────────────
X_train, X_test, Y_train, Y_test, ids_train, ids_test = train_test_split(
    X_encoded, Y, df_model['cve_id'],
    test_size    = TEST_SIZE,
    stratify     = Y,
    random_state = RANDOM_STATE
)

print(f'\n[+] Split : Train={X_train.shape[0]:,}  Test={X_test.shape[0]:,}')
print(f'[+] Y=1 dans Test : {Y_test.sum()} exploitées ({np.mean(Y_test)*100:.2f} %)')

# ── Standardisation pour la régression logistique ─────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## Section 3 — Régression Logistique (Phase 5)

In [ ]:
# ==============================================================================
# CELLULE 4 — Régression Logistique avec et sans correction (Phase 5)
# ==============================================================================

# ── LR sans correction (baseline) ────────────────────────────────────────────
lr_base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, solver='lbfgs')
lr_base.fit(X_train_sc, Y_train)
proba_lr_base = lr_base.predict_proba(X_test_sc)[:, 1]

fpr_b, tpr_b, _ = roc_curve(Y_test, proba_lr_base)
auc_base         = auc(fpr_b, tpr_b)

# ── LR avec class_weight='balanced' ──────────────────────────────────────────
lr_bal = LogisticRegression(class_weight='balanced', max_iter=2000,
                             random_state=RANDOM_STATE, solver='lbfgs')
lr_bal.fit(X_train_sc, Y_train)
proba_lr_bal = lr_bal.predict_proba(X_test_sc)[:, 1]

fpr_bl, tpr_bl, thresh_bl = roc_curve(Y_test, proba_lr_bal)
auc_bal  = auc(fpr_bl, tpr_bl)

# ── Seuil de Youden ───────────────────────────────────────────────────────────
j_bl   = tpr_bl - fpr_bl
tau_bl = thresh_bl[np.argmax(j_bl)]
y_pred_bal = (proba_lr_bal >= tau_bl).astype(int)
cm_lr_bal  = confusion_matrix(Y_test, y_pred_bal)

print('=' * 60)
print('PHASE 5 — RÉGRESSION LOGISTIQUE')
print('=' * 60)
print(f'  AUC — LR sans balance (référence) : {auc_base:.4f}')
print(f'  AUC — LR balanced      (corrigé)  : {auc_bal:.4f}')
print(f'  Gain après correction             : {auc_bal - auc_base:+.4f}')
print()
print(f'  Seuil de Youden τ* (balanced)     : {tau_bl:.6f}')
print(f'  Indice de Youden J(τ*)            : {np.max(j_bl):.4f}')
print()
print(f'  Matrice de confusion — LR balanced (τ* = {tau_bl:.4f}) :')
print(f'  TN={cm_lr_bal[0,0]:4d}  FP={cm_lr_bal[0,1]:4d}')
print(f'  FN={cm_lr_bal[1,0]:4d}  TP={cm_lr_bal[1,1]:4d}')
print()
print(classification_report(Y_test, y_pred_bal,
      target_names=['Non-Exploité', 'Exploité']))

## Section 4 — Random Forest (Phase 6)

In [ ]:
# ==============================================================================
# CELLULE 5 — Random Forest avec balanced_subsample (Phase 6)
# ==============================================================================

rf = RandomForestClassifier(
    n_estimators     = 300,
    class_weight     = 'balanced_subsample',
    max_depth        = 8,
    min_samples_leaf = 3,
    max_features     = 'sqrt',
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)
rf.fit(X_train, Y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

fpr_rf, tpr_rf, thresh_rf = roc_curve(Y_test, proba_rf)
auc_rf   = auc(fpr_rf, tpr_rf)
j_rf     = tpr_rf - fpr_rf
tau_rf   = thresh_rf[np.argmax(j_rf)]
y_pred_rf = (proba_rf >= tau_rf).astype(int)
cm_rf     = confusion_matrix(Y_test, y_pred_rf)

print('=' * 60)
print('PHASE 6 — RANDOM FOREST')
print('=' * 60)
print(f'  AUC-ROC            : {auc_rf:.4f}')
print(f'  Seuil Youden τ*    : {tau_rf:.6f}')
print(f'  Indice Youden J*   : {np.max(j_rf):.4f}')
print(f'  TN={cm_rf[0,0]:4d}  FP={cm_rf[0,1]:4d}')
print(f'  FN={cm_rf[1,0]:4d}  TP={cm_rf[1,1]:4d}')
print()
print(classification_report(Y_test, y_pred_rf,
      target_names=['Non-Exploité', 'Exploité']))

# ── Importance des variables ──────────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_,
                        index=X_encoded.columns).sort_values(ascending=False)
print('\nImportance des variables (Random Forest) :')
print('-' * 50)
for feat, imp in importances.items():
    bar = '█' * int(imp * 100)
    print(f'  {feat:<42} {imp:.4f}  {bar}')

# ── Figure ─────────────────────────────────────────────────────────────────────
fig_imp, ax_imp = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax_imp, color='#1f77b4', edgecolor='black', linewidth=0.5)
ax_imp.set_xlabel('Importance (indice de Gini)', fontsize=11)
ax_imp.set_title('Importance des variables\n(Random Forest — Paradigme 2, CVSS + EPSS)',
                  fontsize=11, fontweight='bold')
ax_imp.invert_yaxis()
plt.tight_layout()
plt.savefig(str(DATA_DIR / 'importance_variables_rf.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print(f'[+] Figure : {DATA_DIR / "importance_variables_rf.pdf"}')

## Section 5 — Benchmark EPSS Standalone (Phase 7)

Comparaison du modèle combiné CVSS+EPSS face à EPSS utilisé seul comme  
classificateur (score EPSS > τ → prédiction Y=1).

> **Note :** cette comparaison n'est pas indépendante — notre modèle *utilise*  
> epss_score comme feature. Elle illustre la contribution marginale de CVSS  
> au-delà du signal EPSS.

In [ ]:
# ==============================================================================
# CELLULE 6 — Benchmark EPSS standalone (Phase 7)
# ==============================================================================

epss_available = False

if epss_ok:
    # ── Téléchargement ou réutilisation du snapshot déjà en mémoire ──────────
    # Le snapshot est déjà dans epss_df si la Cellule 2 a réussi.
    # On ré-télécharge uniquement si epss_df n'est plus disponible.
    try:
        _epss_ref = epss_df.copy()
        print('[+] Réutilisation du snapshot EPSS déjà téléchargé (Cellule 2)')
    except NameError:
        print('[+] Re-téléchargement du snapshot EPSS pour la Phase 7...')
        resp2 = requests.get(EPSS_URL, timeout=60)
        resp2.raise_for_status()
        with gzip.open(io.BytesIO(resp2.content)) as gz2:
            _epss_ref = pd.read_csv(gz2, comment='#', header=0,
                                    names=['cve_id', 'epss_score', 'percentile'])
        _epss_ref['epss_score'] = pd.to_numeric(_epss_ref['epss_score'], errors='coerce')

    # ── Constitution du sous-ensemble de test avec score EPSS ────────────────
    df_eval = pd.DataFrame({
        'cve_id'   : ids_test.values,
        'y_true'   : Y_test,
        'proba_lr' : proba_lr_bal,
        'proba_rf' : proba_rf
    })
    df_eval = df_eval.merge(_epss_ref[['cve_id', 'epss_score']],
                             on='cve_id', how='inner').dropna(subset=['epss_score'])

    n_match  = len(df_eval)
    n_pos    = int(df_eval['y_true'].sum())
    print(f'[+] Sous-ensemble commun : {n_match:,} CVEs appariées ({n_pos} Y=1)')

    if n_pos >= 2:
        fpr_ep, tpr_ep, _ = roc_curve(df_eval['y_true'], df_eval['epss_score'])
        fpr_lr7,tpr_lr7,_ = roc_curve(df_eval['y_true'], df_eval['proba_lr'])
        fpr_rf7,tpr_rf7,_ = roc_curve(df_eval['y_true'], df_eval['proba_rf'])
        auc_ep  = auc(fpr_ep,   tpr_ep)
        auc_lr7 = auc(fpr_lr7,  tpr_lr7)
        auc_rf7 = auc(fpr_rf7,  tpr_rf7)
        epss_available = True

        print()
        print('=' * 60)
        print('PHASE 7 — COMPARAISON SUR SOUS-ENSEMBLE COMMUN')
        print('=' * 60)
        print(f'  AUC LR balanced (notre modèle)   : {auc_lr7:.4f}')
        print(f'  AUC Random Forest (notre modèle) : {auc_rf7:.4f}')
        print(f'  AUC EPSS seul (état de l art)    : {auc_ep:.4f}')
        delta_lr = auc_lr7 - auc_ep
        delta_rf = auc_rf7 - auc_ep
        print(f'  Δ LR vs EPSS seul                : {delta_lr:+.4f}')
        print(f'  Δ RF vs EPSS seul                : {delta_rf:+.4f}')
        print()
        print('  Note : EPSS est entraîné sur un corpus mondial avec des features')
        print('  additionnelles (threat intel, exploit DBs, scans actifs).')
        print('  La comparaison illustre la contribution de CVSS au-delà d EPSS.')
else:
    print('[!] EPSS non disponible — Phase 7 ignorée.')

## Section 6 — Figures ROC Comparatives

In [ ]:
# ==============================================================================
# CELLULE 7 — Figure ROC comparative (Paradigme 2 complet)
# ==============================================================================

fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.3, 1])

# ── Panneau gauche : Courbes ROC ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])

ax1.plot(fpr_b,  tpr_b,  lw=1.5, ls='--', color='#9467bd',
         label=f'LR sans balance   (AUC={auc_base:.3f}) — baseline')
ax1.plot(fpr_bl, tpr_bl, lw=2,   color='#1f77b4',
         label=f'LR balanced       (AUC={auc_bal:.3f})')
ax1.plot(fpr_rf, tpr_rf, lw=2,   color='#2ca02c',
         label=f'Random Forest     (AUC={auc_rf:.3f})')
if epss_available:
    ax1.plot(fpr_ep, tpr_ep, lw=2, ls='-.', color='#d62728',
             label=f'EPSS seul (état art) (AUC={auc_ep:.3f}) *')
ax1.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.4, label='Hasard (AUC=0.500)')

ax1.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax1.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=11)
ax1.set_title('Courbes ROC — Paradigme 2 : CVSS + EPSS\n'
              '(7 CWE Auth/Authz — CISA KEV comme vérité terrain)',
              fontsize=10, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(alpha=0.3)
if epss_available:
    ax1.annotate('* Évalué sur sous-ensemble\ncommun (Phase 7)',
                 xy=(0.55, 0.05), fontsize=8, color='#d62728')

# ── Panneau droit : Tableau récapitulatif ─────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.axis('off')

cm_base_t = confusion_matrix(Y_test, (proba_lr_base >= 0.05).astype(int))
col_labels = ['Modèle', 'AUC-ROC', 'TP', 'FN', 'τ* Youden']
rows = [
    ['LR sans balance\n(GLM baseline)', f'{auc_base:.4f}',
     str(cm_base_t[1,1]), str(cm_base_t[1,0]), '0.050'],
    ['LR balanced', f'{auc_bal:.4f}',
     str(cm_lr_bal[1,1]), str(cm_lr_bal[1,0]), f'{tau_bl:.4f}'],
    ['Random Forest\n(bal. subsample)', f'{auc_rf:.4f}',
     str(cm_rf[1,1]), str(cm_rf[1,0]), f'{tau_rf:.4f}'],
]
if epss_available:
    rows.append(['EPSS seul\n(Jacobs et al. 2021)', f'{auc_ep:.4f}', '—', '—', '—'])

table = ax2.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.1, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#003d6b')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#e8f0fb')
    cell.set_edgecolor('#cccccc')

best_idx = 1 + [auc_base, auc_bal, auc_rf].index(max(auc_base, auc_bal, auc_rf))
for col in range(len(col_labels)):
    table[best_idx, col].set_facecolor('#d4edda')

ax2.set_title('Performances comparées — Paradigme 2', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(str(DATA_DIR / 'comparaison_complete_modeles.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print(f'[+] Figure : {DATA_DIR / "comparaison_complete_modeles.pdf"}')

## Section 7 — Tableau Récapitulatif Final (valeurs à reporter dans le mémoire)

In [ ]:
# ==============================================================================
# CELLULE 8 — Tableau récapitulatif — toutes valeurs pour le mémoire
# ==============================================================================

print('=' * 65)
print('VALEURS FINALES À REPORTER DANS LE MÉMOIRE')
print('=' * 65)
print()

print('DATASET')
print(f'  N total              : {len(df):,}')
print(f'  N train              : {len(X_train):,}')
print(f'  N test               : {len(X_test):,}')
print(f'  Y=1 (exploitées)     : {int(np.sum(Y)):,} total / {Y_test.sum()} dans test '
      f'({np.mean(Y)*100:.2f} %)')
print(f'  Nb features X        : {X_train.shape[1]}'
      f' ({"CVSS + EPSS" if epss_ok else "CVSS seul"})')
print()

if epss_ok:
    print(f'PARADIGME 2 — CVSS + EPSS')
    print(f'  LR sans balance      : AUC = {auc_base:.4f}')
    print(f'  LR balanced          : AUC = {auc_bal:.4f}')
    print(f'  Random Forest        : AUC = {auc_rf:.4f}')
    print()
    print('RÉFÉRENCE PARADIGME 1 — CVSS seul (exécuter Notebook 2 pour les valeurs)')
    print('  (RF balanced_subsample ≈ 0.70 attendu)')
    print()
    best_p2 = max(auc_base, auc_bal, auc_rf)
    print(f'  Δ gain EPSS estimé (P1 ≈ 0.70 → meilleur P2) : '
          f'{best_p2 - 0.7038:+.4f} AUC')
else:
    print('PARADIGME 1 — CVSS seul (EPSS non disponible)')
    print(f'  LR sans balance      : AUC = {auc_base:.4f}')
    print(f'  LR balanced          : AUC = {auc_bal:.4f}')
    print(f'  Random Forest        : AUC = {auc_rf:.4f}')

print()
print('SEUILS DE YOUDEN')
print(f'  LR balanced  τ*      : {tau_bl:.6f}  (J = {np.max(j_bl):.4f})')
print(f'  Random Forest τ*     : {tau_rf:.6f}  (J = {np.max(j_rf):.4f})')
print()
print(f'MATRICE DE CONFUSION — Random Forest (AUC={auc_rf:.4f}, τ*={tau_rf:.4f})')
print(f'  TN={cm_rf[0,0]:4d}  FP={cm_rf[0,1]:4d}')
print(f'  FN={cm_rf[1,0]:4d}  TP={cm_rf[1,1]:4d}')
recall_rf = cm_rf[1,1] / max(cm_rf[1,:].sum(), 1)
prec_rf   = cm_rf[1,1] / max(cm_rf[:,1].sum(), 1)
print(f'  Rappel (Sensibilité)  : {recall_rf:.2%}')
print(f'  Précision             : {prec_rf:.2%}')

if epss_available:
    print()
    print('RÉFÉRENCE EXTERNE — EPSS SEUL (Jacobs et al., 2021)')
    print(f'  AUC EPSS             : {auc_ep:.4f}')
    print(f'  Δ LR vs EPSS seul    : {auc_lr7 - auc_ep:+.4f} AUC')
    print(f'  Δ RF vs EPSS seul    : {auc_rf7 - auc_ep:+.4f} AUC')
print()
print('FIGURES GÉNÉRÉES')
print(f'  {DATA_DIR / "importance_variables_rf.pdf"}')
print(f'  {DATA_DIR / "comparaison_complete_modeles.pdf"}')